In [1]:
!nvidia-smi
!rm -rf /kaggle/working/g1
!git clone -q --branch exp/g1-poc \
  https://github.com/JoaoBornelli/g1_training.git /kaggle/working/g1
!cd /kaggle/working/g1 && git log --oneline -3
!pip install -q mjlab==1.5.3 rsl-rl-lib gymnasium pydantic safetensors
!pip list 2>/dev/null | grep -Ei "^(mjlab|rsl-rl|warp-lang|torch|gymnasium|pydantic|safetensors)"


Thu Aug 20 18:16:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Treino v3

In [2]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/logs/g1_poc --reload_interval 30


<IPython.core.display.Javascript object>

In [3]:
import os
os.environ["WANDB_MODE"] = "disabled"

In [ ]:
import dataclasses, os, pathlib, re, shutil, subprocess, sys

os.environ["WANDB_MODE"] = "disabled"

RAIZ = pathlib.Path("/kaggle/working/g1")        # o clone; o re-clone faz rm -rf aqui
LOG_ROOT = pathlib.Path("/kaggle/working/logs")  # FORA do clone, senão o re-clone apaga o checkpoint
sys.path.insert(0, str(RAIZ))

import g1_poc
from g1_poc.comando import COMANDO_DIM
from g1_poc.curriculo import NIVEL_MAX
from g1_poc.env_cfg import OBS_ATOR, OBS_CRITICO
from mjlab.scripts.train import TrainConfig, launch_training

RUN_DIR = LOG_ROOT / g1_poc.EXPERIMENT / "bloco1"
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ["TORCHRUNX_LOG_DIR"] = str(LOG_ROOT / "torchrunx")

# --- 1. o checkpoint de origem (Add Input), o de MAIOR iteração ---
# ⚠ o número é o PRIMEIRO grupo de dígitos: concatenar tudo faria
# "model_5099_115.pt" virar 5.099.115 e vencer o model_5800 — daí o re.search
# e o filtro de sufixos de largura.
achados = [f for f in pathlib.Path("/kaggle/input").rglob("model_*.pt")
           if "_115" not in f.stem and "_118" not in f.stem]
assert achados, "nenhum model_*.pt em /kaggle/input — confira o Add Input"
achados.sort(key=lambda f: int(re.search(r"model_(\d+)", f.stem).group(1)))
for f in achados:
    print(f"  {f}   ({f.stat().st_size / 2**20:.1f} MB)")
ORIGEM = achados[-1]
BRUTO = RUN_DIR / ORIGEM.name
shutil.copy(ORIGEM, BRUTO)

# (sem bump de std desta vez: o model_5800 já carrega o std aprendido ~0,49)

# --- 2. a CIRURGIA para o contrato ATUAL: ator 112|115 -> 118, crítico 125 -> 128 ---
# `face_normal_b` (20/08) e `base_lin_vel` de volta ao ator (20/08, §5.1 revertida
# com gatilho medido: 530 iterações com 30% de dados de andar e a duração caiu
# 112 -> 26 passos). O script FALHA se não mudar nada — cópia intacta silenciosa
# foi o modo de falha de 20/08.
EXPANDIDO = BRUTO.with_name(BRUTO.stem + "_118.pt")
subprocess.run([sys.executable, "-m", "g1_poc.expande_checkpoint",
                "--entrada", str(BRUTO), "--saida", str(EXPANDIDO)],
               cwd=RAIZ, check=True)
# ⚠ o resume resolve `model_.*` por ordem; a largura velha NÃO pode sobrar no run dir
BRUTO.unlink()
CHECKPOINT = EXPANDIDO.name
print(f"\ncheckpoint pronto = {EXPANDIDO}")

In [ ]:
# LOG_ROOT, RUN_DIR e CHECKPOINT vêm da célula anterior. Não redigitar.
TASK = g1_poc.TASK_ID                # "Mjlab-G1-Poc"

cfg = TrainConfig.from_task(TASK)
cfg = dataclasses.replace(cfg, log_root=str(LOG_ROOT))

cfg.env.scene.num_envs = 4096            # POR RANK
cfg.agent.max_iterations = 12000         # ⚠ ADICIONAIS ao contador do checkpoint
cfg.agent.resume = True
cfg.agent.load_run = RUN_DIR.name        # "bloco1", explícito
cfg.agent.load_checkpoint = CHECKPOINT   # o _118 da célula anterior
cfg.agent.run_name = "bloco1"            # o dir novo sai <timestamp>_bloco1
cfg.agent.algorithm.learning_rate = 5e-4  # warm-start SEMPRE 5e-4 (lição do repo)

# ---- CONFERE. Mutação silenciosa que não pega queima a sessão. ----
_cmds = list(cfg.env.commands)
_ator = cfg.env.observations["actor"].terms
_critico = cfg.env.observations["critic"].terms
_curr = list(cfg.env.curriculum)
_tarefa = ("staged", "precise_pos", "precise_ori", "squeeze", "unload",
           "postura_ereta", "sustentacao", "load", "joint_vel_hinge")

print(f"[CONFERE] comando        = {COMANDO_DIM} canais, ordem {_cmds}")
print(f"[CONFERE] recompensas    = {len(cfg.env.rewards)}  (13 fundação + self_collisions + 9 tarefa)")
print(f"[CONFERE] terminações    = {len(cfg.env.terminations)}  {sorted(cfg.env.terminations)}")
print(f"[CONFERE] currículo      = {_curr}  (twist_ranges e nivel ANTES de forma)")
print(f"[CONFERE] obs alvo       = ator {OBS_ATOR} / crítico {OBS_CRITICO} canais")
print(f"[CONFERE] último do ator = {list(_ator)[-1]}  (canal novo por último = contrato do append)")
print(f"[CONFERE] base_lin_vel   = ator {'base_lin_vel' in _ator} (ruidoso) / crítico {'base_lin_vel' in _critico} (limpo)  — esperado True/True")
print(f"[CONFERE] NIVEL_MAX      = {NIVEL_MAX}")
print(f"[CONFERE] learning_rate  = {cfg.agent.algorithm.learning_rate}")
print(f"[CONFERE] load_run       = {cfg.agent.load_run}   checkpoint = {cfg.agent.load_checkpoint}")
print(f"[CONFERE] log_root       = {cfg.log_root}")

assert COMANDO_DIM == 10
assert _cmds[0] == "caixa_alvo", f"`caixa_alvo` tem de vir ANTES do twist: {_cmds}"
assert len(cfg.env.rewards) == 23, f"esperado 23, medido {len(cfg.env.rewards)} — código velho no clone?"
assert len(cfg.env.terminations) == 4
assert all(n in cfg.env.rewards for n in _tarefa), "faltou termo de tarefa"
assert (OBS_ATOR, OBS_CRITICO) == (118, 128), "contrato de obs errado — código velho?"
assert list(_ator)[-1] == "base_lin_vel", "o canal mais novo tem de ser o ÚLTIMO"
assert _ator["base_lin_vel"].noise is not None, "o base_lin_vel do ator tem de ser RUIDOSO (±0,5)"
assert _curr.index("twist_ranges") < _curr.index("forma") and _curr.index("nivel") < _curr.index("forma"), \
    f"ordem do currículo errada ({_curr}) — a promoção leria a forma do episódio SEGUINTE"
assert cfg.env.terminations["time_out"].time_out is True, "o rsl_rl trataria time_out como fracasso"
assert "base_lin_vel" in _ator and "base_lin_vel" in _critico
assert "base_com" not in cfg.env.events, "dr.body_com_offset corrompe a heap"
assert cfg.env.rewards["dof_pos_limits"].weight == -10.0
assert NIVEL_MAX == 6
assert cfg.agent.experiment_name == "g1_poc", "experiment errado casa a run de outro pacote no resume"
assert cfg.agent.resume is True and cfg.agent.load_checkpoint.endswith("_118.pt")
assert not str(LOG_ROOT).startswith(str(RAIZ)), "log dentro do repo: o re-clone apaga o checkpoint"

launch_training(TASK, cfg)

---

In [ ]:
# import subprocess
# print(subprocess.run(
#     ['python', '-c', 'import torch;'
#      'print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),'
#      '"|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")'],
#     cwd='/kaggle/working/g1', capture_output=True, text=True).stdout)

In [ ]:
# !cd /kaggle/working/g1 && python g1_multitask/smoke.py

In [ ]:
# Opcional
# !cd /kaggle/working/g1 && python g1_multitask/preflight_gpu.py 2048

# Treinamento do zero

In [ ]:
# import dataclasses, os, pathlib, sys

# RAIZ = pathlib.Path('/kaggle/working/g1')
# sys.path.insert(0, str(RAIZ))

# # Sem isto, no DDP as linhas [CURRICULO] vão pra arquivo e somem da célula.
# os.environ["TORCHRUNX_LOG_DIR"] = str(RAIZ / "logs" / "torchrunx")

# import g1_multitask
# from g1_multitask import tasks as T
# from mjlab.scripts.train import TrainConfig, launch_training

# TASK = g1_multitask.TASK_ID          # "Mjlab-Multitask-Unitree-G1"
# LOG_ROOT = RAIZ / "logs"

# cfg = TrainConfig.from_task(TASK)
# cfg = dataclasses.replace(cfg, log_root=str(LOG_ROOT))

# cfg.env.scene.num_envs = 4096        # POR RANK. O bloco 1 rodou 4096 num T4 só.
# cfg.agent.max_iterations = 2500
# cfg.agent.resume = False             # bloco 1: do zero. A obs mudou de tarefa.

# # ---- CONFERE. Mutação silenciosa que não pega queima a sessão. ----
# _tw = cfg.env.commands["twist"]
# _post = [n for n in cfg.env.rewards if n.startswith("post")]
# print(f"[CONFERE] tarefas        = {T.NUM_TASKS}  ({list(T.NAMES.values())})")
# print(f"[CONFERE] destravamentos = {T.total_unlocks()}")
# print(f"[CONFERE] heading_command= {_tw.heading_command}  stiffness={_tw.heading_control_stiffness}")
# print(f"[CONFERE] piso do giro   = {_tw.piso_giro_rad_s}  (fatia {_tw.frac_giro_no_standing} do standing {_tw.rel_standing_envs})")
# print(f"[CONFERE] postura        = {_post}")
# print(f"[CONFERE] fora_da_area   = {'fora_da_area' in cfg.env.terminations}  (esperado False)")
# print(f"[CONFERE] experiment     = {cfg.agent.experiment_name}")

# assert T.NUM_TASKS == 5 and T.total_unlocks() == 24
# assert _tw.heading_command and _tw.ranges.heading is not None, "heading inerte em silêncio"
# assert _tw.piso_giro_rad_s > 0.05, "piso abaixo do command_threshold da marcha"
# assert len(_post) == 1, "a postura tem de ser UM termo"
# assert "fora_da_area" not in cfg.env.terminations
# assert cfg.agent.experiment_name == "g1_multitask"

# launch_training(TASK, cfg)

# Resume

In [ ]:
# import dataclasses, os, pathlib, sys

# RAIZ = pathlib.Path('/kaggle/working/g1')
# sys.path.insert(0, str(RAIZ))

# # Sem isto, no DDP as linhas [CURRICULO] vão pra arquivo e somem da célula.
# os.environ["TORCHRUNX_LOG_DIR"] = str(LOG_ROOT / "torchrunx")

# import g1_multitask
# from g1_multitask import tasks as T
# from mjlab.scripts.train import TrainConfig, launch_training
# from mjlab.utils.os import get_checkpoint_path

# TASK = g1_multitask.TASK_ID

# cfg = TrainConfig.from_task(TASK)
# # LOG_ROOT, RUN_DIR e CHECKPOINT vêm da célula do checkpoint. Não redigitar.
# cfg = dataclasses.replace(cfg, log_root=str(LOG_ROOT))

# cfg.env.scene.num_envs = 4096
# cfg.agent.max_iterations = 5000
# """⚠️ ADICIONAIS, não total. O rsl_rl faz `total_it = start_it + num_learning_iterations`
# (`on_policy_runner.py:77-78`) e o `load` restaura o contador do checkpoint (`:16`).
# Retomando de 2300 com 2500 aqui, a run vai até 4800."""

# cfg.agent.resume = True
# cfg.agent.load_run = RUN_DIR.name         # nome EXPLÍCITO, não ".*"
# cfg.agent.load_checkpoint = 'model_17297.pt'   # CHECKPOINT
# cfg.agent.algorithm.learning_rate = 5e-4 

# # ---- 1. o checkpoint RESOLVE? Antes de montar o env. ----
# # ⚠️ O mjlab constrói o env em `train.py:110` e só resolve o resume em `:133`. Um
# # caminho errado custa os 2 a 3 min de montagem e só então levanta.
# _ck = get_checkpoint_path(LOG_ROOT / cfg.agent.experiment_name,
#                           cfg.agent.load_run, cfg.agent.load_checkpoint)
# print(f"[CONFERE] retoma de      = {_ck}")
# assert _ck.is_file(), f"resolveu para algo que não é arquivo: {_ck}"

# # ---- 2. o desenho é o mesmo que o bloco 1 treinou? ----
# _tw = cfg.env.commands["twist"]
# _post = [n for n in cfg.env.rewards if n.startswith("post")]
# _tol = cfg.env.metrics["sucesso"].params["tol"]
# print(f"[CONFERE] tarefas        = {T.NUM_TASKS}  ({list(T.NAMES.values())})")
# print(f"[CONFERE] destravamentos = {T.total_unlocks()}")
# print(f"[CONFERE] heading        = {_tw.heading_command}  stiffness={_tw.heading_control_stiffness}")
# print(f"[CONFERE] piso do giro   = {_tw.piso_giro_rad_s}")
# print(f"[CONFERE] postura        = {_post}")
# print(f"[CONFERE] tolerancias    = tol_v={_tol.tol_v}  tol_w={_tol.tol_w}")
# print(f"[CONFERE] iteracoes      = +{cfg.agent.max_iterations} adicionais")

# assert T.NUM_TASKS == 5 and T.total_unlocks() == 24
# assert _tw.heading_command and _tw.ranges.heading is not None, "heading inerte em silêncio"
# assert _tw.piso_giro_rad_s > 0.05, "piso abaixo do command_threshold da marcha"
# assert len(_post) == 1, "a postura tem de ser UM termo"
# assert "fora_da_area" not in cfg.env.terminations
# assert cfg.agent.experiment_name == "g1_multitask"

# # ---- 3. OPCIONAL: afrouxar a régua do sucesso ----
# # Na iteração 1934 o bloco 1 mediu erro_lin 0,22 contra tol_v 0,25 e erro_ang 0,34
# # contra tol_w 0,35, com perf_n0 = 0,51. As duas médias estão COLADAS no limiar, e
# # média no limiar significa metade dos episódios passando. O portão exige 0,90.
# #
# # O §8 registra que o tol_v foi fixado SEM medição ("risco aceito, decisão do dono
# # do experimento"). Agora existe medição.
# #
# # ⚠️ Mudar limiar de sucesso é Categoria C: a régua se move e a EMA acumulada passa
# # a significar outra coisa. Com perf_n0 em 0,51 o custo é baixo, mas o número deixa
# # de ser comparável com o do bloco 1. Descomente por decisão, não por reflexo.
# #
# # _tol.tol_v = 0.35
# # _tol.tol_w = 0.50
# # print(f"[CONFERE] tolerancias AFROUXADAS = tol_v={_tol.tol_v}  tol_w={_tol.tol_w}")

# launch_training(TASK, cfg)

# Depois do bloco


In [ ]:
# import pathlib, subprocess
# from IPython.display import Image, display

# RAIZ_LOG = pathlib.Path('/kaggle/working/logs/g1_multitask')
# runs = [d for d in RAIZ_LOG.iterdir() if d.is_dir() and d.name != "bloco1"]
# assert runs, f"nenhuma run em {RAIZ_LOG}"
# RUN = sorted(runs, key=lambda d: d.stat().st_mtime)[-1]
# print("run:", RUN.name, "\n")

# # ⚠️ O caminho é POSICIONAL e é o diretório da RUN, não o do experimento
# # (`entre_blocos.py:391-394`). O default dele é `<repo>/logs`, que não existe —
# # o log mora fora do repo para o clone não apagá-lo.
# r = subprocess.run(["python", "g1_multitask/entre_blocos.py", str(RUN), "--grafico"],
#                    cwd="/kaggle/working/g1", capture_output=True, text=True)
# print(r.stdout or r.stderr)

# for png in sorted(pathlib.Path('/kaggle/working/g1').glob('painel_*.png')):
#     display(Image(str(png)))

In [ ]:
# import pathlib, shutil

# raiz = pathlib.Path('/kaggle/working/logs/g1_multitask')
# runs = [d for d in raiz.iterdir() if d.is_dir() and d.name != "bloco1"]
# run = sorted(runs, key=lambda d: d.stat().st_mtime)[-1]
# ckpts = sorted(run.glob('model_*.pt'), key=lambda f: int(f.stem.split('_')[1]))
# print('run:', run.name)
# for c in ckpts[-3:]:
#     print(f'  {c.name}   {c.stat().st_size / 2**20:.1f} MB')

# shutil.copy(ckpts[-1], f'/kaggle/working/{ckpts[-1].name}')
# print(f'\nbaixe /kaggle/working/{ckpts[-1].name} pelo painel de Output')